In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/StoichModelGP/ModelGP_RBF.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 1.8158454655635485, 'n_it': 1.2301065099118285}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=3)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}
            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.121438580721229

Trial 1 =========================================
15.132828861136861

Trial 2 =========================================
15.126594326810851

Trial 3 =========================================
15.138342405567618

Trial 4 =========================================
15.141772412741053

Trial 5 =========================================
15.13243412413727

Trial 6 =========================================
15.142302608133795

Trial 7 =========================================
15.142370909976385

Trial 8 =========================================
15.1341322706477

Trial 9 =========================================
15.14334489122013

Trial 10 =========================================
15.119830659944782

Trial 11 =========================================
15.140787469894423

Trial 12 =========================================
15.125823148432463

Trial 13 =========================================
15.131414968821316

Trial 14 ===========

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 24 =========================================
15.140057268236275

Trial 25 =========================================
15.135987406603228

Trial 26 =========================================
15.133960797851088

Trial 27 =========================================
15.131053548674327

Trial 28 =========================================
15.138833659421888

Trial 29 =========================================
15.140542135635533



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 30 =========================================
15.131431500303037



...

Trial 48 =========================================
15.132179720121002

Trial 49 =========================================
15.137350760169095

Trial 50 =========================================
15.106088406838014

Trial 51 =========================================
15.137809071978793

Trial 52 =========================================
15.127285632005028

Trial 53 =========================================
15.138675128025213

Trial 54 =========================================
15.138213910277177

Trial 55 =========================================
15.136884399904345

Trial 56 =========================================
15.138502702494769

Trial 57 =========================================
15.135812806588724

Trial 58 =========================================
15.142293219023333

Trial 59 =========================================
15.138938118980263

Trial 60 =========================================
15.143422173266806

Trial 61 =========================================
15.135627925157475

Trial 

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 78 =========================================
15.130704141234919

Trial 79 =========================================
15.138947842463375

Trial 80 =========================================
15.135144829884824

Trial 81 =========================================
15.121355956646173

Trial 82 =========================================
15.142770044019962

Trial 83 =========================================
15.138103969885218



/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 84 =========================================
15.133149865068635

Trial 85 =========================================
15.136876284009304

Trial 86 =========================================
15.134851806907006

Trial 87 =========================================
15.133052535836056

Trial 88 =========================================
15.139047482205736

Trial 89 =========================================
15.125432377781241

Trial 90 =========================================
15.126746436567299

Trial 91 =========================================
15.141499475121186

Trial 92 =========================================
15.132812260836811

Trial 93 =========================================
15.14185329486749

Trial 94 =========================================
15.138841822446224

Trial 95 =========================================
15.120533726541847

Trial 96 =========================================
15.141553811280819

Trial 97 =========================================
15.135604450772172

Trial 9

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 15.143507426335884
Avg = 15.135141664867014
Std = 0.007669257187496908


In [7]:
print(y_max_arr.tolist())

[15.121438580721229, 15.132828861136861, 15.126594326810851, 15.138342405567618, 15.141772412741053, 15.13243412413727, 15.142302608133795, 15.142370909976385, 15.1341322706477, 15.14334489122013, 15.119830659944782, 15.140787469894423, 15.125823148432463, 15.131414968821316, 15.136139724886656, 15.142001772016538, 15.112411434652783, 15.140579648045248, 15.143031116322142, 15.124600024105128, 15.136261863628643, 15.141664169252525, 15.141494292620948, 15.143507426335884, 15.140057268236275, 15.135987406603228, 15.133960797851088, 15.131053548674327, 15.138833659421888, 15.140542135635533, 15.131431500303037, 15.137051086531983, 15.135316111549932, 15.139164795349302, 15.142184669239182, 15.134653207777772, 15.140847824509748, 15.143350247376983, 15.132077193952444, 15.119525690372454, 15.14323153609743, 15.12373303739899, 15.133223050456557, 15.119123503257853, 15.142474368424619, 15.133353940445781, 15.142422970035376, 15.142615377785578, 15.132179720121002, 15.137350760169095, 15.10

In [8]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_RBF/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/TestPredModelGP_RBF/DataGenerated/normal_EI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    15.139807
1    15.133235
2    15.138981
3    15.141145
4    15.110410
..         ...
295  15.120534
296  15.141554
297  15.135604
298  15.137991
299  15.137684

[300 rows x 1 columns]
